In [2]:
Here are the three concrete deliverables for **Exp 22**.

---

### 1. Pseudocode / Function Signature for the Confrontation Dream Sampler

```python
def run_dream_phase(
    agent,
    n_dream_trials: int,
    mode: str,                    # "replacement" | "confrontation"
    old_patterns: list,
    new_patterns: list,
    old_zero_fraction: float = 0.5,   # only used in confrontation
    zero_reward_type: str = "omission",  # "omission" | "explicit_wrong"
    interleave: bool = True,
    log_v: bool = True
) -> dict:
    """
    Offline replay phase.
    
    Returns:
        {
            "old_mass_trajectory": list[float],
            "new_mass_trajectory": list[float],
            "rpe_old": list[float],
            "crossover_trial": int | None,
            "final_old_mass": float,
            "final_new_mass": float
        }
    """
    old_traj, new_traj, rpe_old = [], [], []
    crossover_trial = None

    for t in range(n_dream_trials):
        if mode == "replacement":
            pattern = sample(new_patterns)
            reward = high_reward(pattern)          # normal new reward
        else:  # confrontation
            if np.random.rand() < old_zero_fraction:
                pattern = sample(old_patterns)
                reward = 0.0 if zero_reward_type == "omission" else explicit_wrong_reward()
            else:
                pattern = sample(new_patterns)
                reward = high_reward(pattern)

        # Core update (same rule as real experience)
        rpe = agent.update(pattern, reward)        # returns RPE on the active V-stock

        # Logging
        old_mass = agent.get_mass(old_patterns)
        new_mass = agent.get_mass(new_patterns)
        old_traj.append(old_mass)
        new_traj.append(new_mass)
        if pattern in old_patterns:
            rpe_old.append(rpe)

        if crossover_trial is None and new_mass > old_mass:
            crossover_trial = t

    return {
        "old_mass_trajectory": old_traj,
        "new_mass_trajectory": new_traj,
        "rpe_old": rpe_old,
        "crossover_trial": crossover_trial,
        "final_old_mass": old_traj[-1],
        "final_new_mass": new_traj[-1]
    }
```

Key implementation notes:
- `agent.update()` must be the **exact same** value-update rule used in real experience (so the only difference is the sampling distribution and forced zero reward).
- Force the reward to exactly 0 (or a fixed negative) for old patterns; do not let the environment’s normal reward function run.
- Log both masses every dream trial — this is the most diagnostic output.

---

### 2. Results Table Template (ready to fill)

**Exp 22 — Summary Metrics** (mean ± SEM across seeds)

| Arm | Old mass | New mass | Superstition | Energy | Crossover | ΔOld during dream | Peak neg. RPE (old) |
|-----|----------|----------|--------------|--------|-----------|-------------------|---------------------|
| 1: Real only | | | | | never | — | — |
| 2: Replacement | | | | | never / late | ≈ 0 | ≈ 0 |
| 3: Confrontation | | | | | **inside dream** | **negative** | **large** |
| 4: Confrontation + gas | | | | | **inside dream (faster)** | **more negative** | large |

**Crossover Latency** (dream trials to crossover, or “never”)

| Arm | Median | 25th–75th | % of seeds that crossed |
|-----|--------|-----------|-------------------------|
| 2 | never | — | 0 % |
| 3 | | | |
| 4 | | | |

---

### 3. Figure Plan

**Figure 1 — Mass Trajectories (most important)**  
- X-axis: dream trial number  
- Y-axis: mass  
- Two lines per panel: Old mass (solid) and New mass (dashed)  
- Four panels side-by-side (or 2×2): Arm 1 (flat), Arm 2 (new rises, old flat), Arm 3 (old declines, new rises, crossover), Arm 4 (same but steeper old decline)  
- Mean ± SEM shading  
- Vertical dashed line at the median crossover trial for Arms 3/4

**Figure 2 — Summary Bars**  
- Grouped bar plot of final Old mass, New mass, Superstition, Energy for the four arms  
- Error bars = SEM

**Figure 3 — Demolition Diagnostics**  
- Left: histogram or violin of ΔOld during dream (Arm 2 vs 3 vs 4)  
- Right: peak negative RPE on old-pattern presentations (Arm 3 vs 4)

**Figure 4 (optional)** — Survival curves of “time to crossover” for Arms 2–4

---

These three pieces are ready to drop straight into the notebook.  

Do you want me to also write the full experiment runner skeleton (acquisition → shift → dream → probe) that calls the sampler above, or shall we first lock the exact numbers (trial counts, fractions, seeds) from Exp 21 so everything matches 1:1?

SyntaxError: invalid character '’' (U+2019) (3871853005.py, line 73)